### Using KnowRob in Python



This notebook demonstrates how to use the KnowRob system directly in Python. It includes importing necessary modules, initializing the knowledge base, and executing queries.

### Importing KnowRob Modules

In [ ]:
import json
from knowrob import *

from pycrap.ontologies import is_exerted_by

First, we import the required modules from KnowRob. The `try-except` block ensures compatibility with different ROS environments, either using the ROS1-specific package or directly loading `knowrob.so`.

In [ ]:
InitKnowRob()

The `InitKnowRob()` function initializes the KnowRob system, setting up necessary configurations and connections.

### Setting Up Knowledge Base

In [ ]:
# Sample dictionary to be converted to JSON
sample_dict = {
	"logging": {
		"console-sink": {"level": "debug"},
		"file-sink": {"level": "debug"}
	},
	"semantic-web": {
		"prefixes": [
			{"alias": "swrl_test", "uri": "http://knowrob.org/kb/swrl_test"}
            # {"alias": "pizza", "uri": "http://www.co-ode.org/ontologies/pizza/pizza.owl"}
		]
	},
	"data-sources": [
		{"path": "tests/owl/swrl.owl", "format": "rdf-xml"}
        # {"path": "tests/owl/pizza.owl", "format": "rdf-xml"}
	],
	"data-backends": [
		{
			"type": "MongoDB",
			"name": "mongodb",
			"host": "localhost",
			"port": 27017,
			"db": "swrl2",
			"read-only": False
		}
	],
	"reasoner": [
    ]
}
# Convert the dictionary to a JSON string
json_str = json.dumps(sample_dict)
# Initialize the KnowledgeBase with the PropertyTree
kb = KnowledgeBase(json_str)

This block defines the configuration for the KnowledgeBase, including logging, semantic web prefixes, data sources, and backends. The configuration is then serialized to a JSON string and used to initialize the `KnowledgeBase` instance.

### Submitting a Query

In [5]:
phi1 = QueryParser.parse("swrl_test:hasAncestor(swrl_test:'Fred', ?y)")
# phi1 = QueryParser.parse("pizza:hasCountryOfOrigin(pizza:'AmericanSlicer', ?y)")
# phi1 = QueryParser.parse("pizza:hasCountryOfOrigin(pizza:'Mozarella', ?y)")

Here, a query is parsed using the `QueryParser`. The query checks for ancestors of the entity `Lea` within the `swrl_test` namespace.

### Retrieving Query Results

In [6]:
resultStream = kb.submitQuery(phi1, QueryContext(QueryFlag.QUERY_FLAG_ALL_SOLUTIONS))
resultQueue = resultStream.createQueue()
# Get the result
nextResult1 = resultQueue.pop_front()

The query formulated in the previous step is submitted to the KnowledgeBase. The results are retrieved as a stream, and a queue is created to handle them.

### Processing Query Results


In [7]:
if isinstance(nextResult1, AnswerYes):
    for substitution in nextResult1.substitution():
        variable = substitution[1]
        term = substitution[2]
        print(str(variable) + " : " + str(term))

?y : swrl_test:Rex


This block checks if the result is affirmative (`AnswerYes`) and prints each substitution found in the query result, listing variable bindings.

### Negative Query Result Handling


In [ ]:
phi2 = QueryParser.parse("swrl_test:hasSibling(swrl_test:'Ernest', swrl_test:'Fred')")
resultStream = kb.submitQuery(phi2, QueryContext(QueryFlag.QUERY_FLAG_ALL_SOLUTIONS))
resultQueue = resultStream.createQueue()
# Get the result
nextResult2 = resultQueue.pop_front()
if isinstance(nextResult2, AnswerNo):
    print("result is negative")
else:
    print("result is positive")

A second query checks for a specific condition, in this case, whether `Lea` is an ancestor of herself, which is expected to be false. The result is handled accordingly.

### Inconclusive Query Result Handling

In [ ]:
phi3 = QueryParser.parse("r(?x, ?y)")
resultStream = kb.submitQuery(phi3, QueryContext(QueryFlag.QUERY_FLAG_ALL_SOLUTIONS))
resultQueue = resultStream.createQueue()
# Get the result
nextResult3 = resultQueue.pop_front()
if isinstance(nextResult3, AnswerDontKnow):
    print("We can't say if the result is true or false")


The final example demonstrates handling a situation where the system cannot determine the truth value of the query, resulting in an `AnswerDontKnow` response.

### Ontology Reasoner

In [ ]:
import json
from knowrob import *
InitKnowRob()

In [ ]:
# Sample dictionary to be converted to JSON
sample_dict = {
	"logging": {
		"console-sink": {"level": "debug"},
		"file-sink": {"level": "debug"}
	},
	"semantic-web": {
		"prefixes": [
			{"alias": "swrl_test", "uri": "http://knowrob.org/kb/swrl_test"},
            # {"alias": "pizza", "uri": "http://www.co-ode.org/ontologies/pizza/pizza.owl"}
            {"alias": "nlquery", "uri": "http://knowrob.org/kb/nlquery"}
		]
	},
	"data-sources": [
		{"path": "tests/owl/swrl.owl", "format": "rdf-xml"}
        # {"path": "tests/owl/pizza.owl", "format": "rdf-xml"}
	],
	"data-backends": [
		{
			"type": "MongoDB",
			"name": "mongodb",
			"host": "localhost",
			"port": 27017,
			"db": "swrl",
			"read-only": False
		}
	],
	"reasoner": [
        {
            "name": "OntoQueryReasoner",
            "type": "OntoQueryReasoner",
            "module": "/home/malineni/ROS_WS/knowrob/tutorials/OntoReasoner.py",
			"data-backend": "mongodb",
        }
    ]
}
# Convert the dictionary to a JSON string
json_str = json.dumps(sample_dict)
# Initialize the KnowledgeBase with the PropertyTree
kb = KnowledgeBase(json_str)

In [ ]:
phi1 = QueryParser.parse('nlquery:nlquery("get the child classes of the Locomotion class", ?response)')
phi1

In [ ]:
resultStream = kb.submitQuery(phi1, QueryContext(QueryFlag.QUERY_FLAG_ALL_SOLUTIONS))
resultQueue = resultStream.createQueue()
# Get the result
nextResult1 = resultQueue.pop_front()

In [ ]:
type(nextResult1)

In [ ]:
if isinstance(nextResult1, AnswerYes):
    for substitution in nextResult1.substitution():
        variable = substitution[1]
        term = substitution[2]
        print(str(variable) + " : " + str(term))

In [ ]:
for bind in nextResult1.substitution():
    variable = bind[1]
    term = bind[2]

In [ ]:
print(term)

### ActionDesignator Reasoner

In [1]:
import json
from knowrob import *
InitKnowRob()

[10:34:52.565] [info] [KnowRob] static initialization done.


In [2]:
# Sample dictionary to be converted to JSON
sample_dict = {
	"logging": {
		"console-sink": {"level": "debug"},
		"file-sink": {"level": "debug"}
	},
	"semantic-web": {
		"prefixes": [
			{"alias": "swrl_test", "uri": "http://knowrob.org/kb/swrl_test"},
            {"alias": "nlquery", "uri": "http://knowrob.org/kb/nlquery"}
		]
	},
	"data-sources": [
		{"path": "tests/owl/swrl.owl", "format": "rdf-xml"}
	],
	"data-backends": [
		{
			"type": "MongoDB",
			"name": "mongodb",
			"host": "localhost",
			"port": 27017,
			"db": "swrlad",
			"read-only": False
		}
	],
	"reasoner": [
        {
            "name": "ADReasoner",
            "type": "ADReasoner",
            "module": "/home/malineni/ROS_WS/knowrob/tutorials/ActionDesignatorReasoner.py",
			"data-backend": "mongodb",
        }
    ]
}
# Convert the dictionary to a JSON string
json_str = json.dumps(sample_dict)
# Initialize the KnowledgeBase with the PropertyTree
kb = KnowledgeBase(json_str)
phi1 = QueryParser.parse('nlquery:nlquery("cut the brocolli using the big sharp black knife on the desk", ?response)')
phi1

[10:35:46.862] [info] Using backend `mongodb` with type `MongoDB`.
[10:35:46.873] [info] [mongodb] connected to mongodb://localhost:27017 (swrlad.triples).
[10:35:47.768] [info] Using queryable backend with id 'mongodb'.
[10:35:47.768] [info] Using persistent backend with id 'mongodb'.
[10:35:47.774] [info] Loading ontology at '/home/malineni/ROS_WS/knowrob/owl/rdf-schema.xml' with version "Tue Jun 11 11:53:20 2024" and origin "rdf-schema".
[10:35:47.791] [info] Loading ontology at '/home/malineni/ROS_WS/knowrob/owl/owl.rdf' with version "Tue Jun 11 11:53:20 2024" and origin "owl".
[10:35:47.798] [info] Loading ontology at '/home/malineni/ROS_WS/knowrob/tests/owl/swrl.owl' with version "Tue Jan 14 12:33:30 2025" and origin "swrl".
[10:35:47.842] [info] Using reasoner `ADReasoner` with type `ADReasoner`.
[10:35:47.842] [info] Using goal-driven reasoner with id 'ADReasoner'.


nlquery:nlquery("cut the brocolli using the big sharp black knife on the desk", ?response)

In [3]:
resultStream = kb.submitQuery(phi1, QueryContext(QueryFlag.QUERY_FLAG_ALL_SOLUTIONS))
resultQueue = resultStream.createQueue()
# Get the result
nextResult1 = resultQueue.pop_front()

[10:36:02.070] [error] LLM query failed: HTTPConnectionPool(host='127.0.0.1', port=5000): Max retries exceeded with url: /query (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7fdc7029ed60>: Failed to establish a new connection: [Errno 111] Connection refused'))
[10:36:02.070] [error] Failed to get response from LLM
[10:36:02.070] [warning] Reasoner 'ADReasoner' produced 'false' in query evaluation for query: ∧ nlquery:nlquery("cut the brocolli using the big sharp black knife on the desk", ?response)


In [4]:
if isinstance(nextResult1, AnswerYes):
    for substitution in nextResult1.substitution():
        variable = substitution[1]
        term = substitution[2]
        print(str(variable) + " : " + str(term))

### PyCRAPOntology

In [2]:
from src.pycrap.ontology_wrapper import OntologyWrapper

In [3]:
ow = OntologyWrapper()

In [4]:
ow.ontology

get_ontology("file:///tmp/tmpwbr51xhw#")

In [5]:
ow.python_objects

{}

In [6]:
ow.file

In [7]:
ow.classes()

<generator object _GraphManager.classes at 0x7fe95a10c740>

In [8]:
[cls for cls in ow.classes()]

[PyCRAP.Base,
 PyCRAP.PhysicalAttribute,
 PyCRAP.ForceAttribute,
 PyCRAP.FrictionAttribute,
 PyCRAP.NetForce,
 PyCRAP.StaticFrictionAttribute,
 PyCRAP.KineticFrictionAttribute,
 PyCRAP.ForceMinimalExertion,
 PyCRAP.ForceProprioceptionFeedback,
 PyCRAP.ForceMinorResistance,
 PyCRAP.ForceContactForce,
 PyCRAP.ForceResistanceFromObject,
 PyCRAP.ForceEquilibriumHold,
 PyCRAP.ForceEquilibriumState,
 PyCRAP.ForceGravityCompensation,
 PyCRAP.ForcePreloading,
 PyCRAP.ForceShearForce,
 PyCRAP.ForceCompressiveForce,
 PyCRAP.ForceBlockageEvent,
 PyCRAP.ForceBreakageEvent,
 PyCRAP.ForceModulationDuringCutting,
 PyCRAP.ForceLoadReduction,
 PyCRAP.ForceEquilibriumReset,
 PyCRAP.ForceFrictionGrip,
 PyCRAP.ForceFrictionResistance,
 PyCRAP.ForceSeparationEvent,
 PyCRAP.World,
 PyCRAP.Floor,
 PyCRAP.Milk,
 PyCRAP.Robot,
 PyCRAP.Cereal,
 PyCRAP.Kitchen,
 PyCRAP.PouringTool,
 PyCRAP.Food,
 PyCRAP.Apartment,
 PyCRAP.Cup,
 PyCRAP.Spoon,
 PyCRAP.Bowl,
 PyCRAP.PreferredGraspAlignment,
 PyCRAP.XAxis,
 PyCRAP.Y

In [9]:
[ind for ind in ow.individuals()]

[]

In [10]:
ow.reason(save_to="inferred_SOMA.owl")

In [17]:
ow.python_objects

{}

In [2]:
from owlready2 import *

In [3]:
onto = get_ontology("SOMA.owl").load()

In [4]:
with onto:
    class ForceMinimalExertion(onto.ForceAttribute):
        pass
    class ForceMinorResistance(onto.ForceAttribute):
        pass


In [8]:
with onto:
    class exerts_force(ObjectProperty):
        domain = [onto.ForceMinimalExertion]
        range = [ForceMinorResistance]
        transitive = True

In [11]:
with onto:
    class is_exerted_by(ObjectProperty):
        domain = [onto.ForceAttribute]
        range = [onto.PhysicalObject]
        transitive = True

AttributeError: 'NoneType' object has no attribute 'storid'

In [14]:
# onto.save(file="extended_ontology.owl", format="rdfxml")

In [10]:
type(onto.ForceAttribute)

owlready2.entity.ThingClass